# Task 2.5 — SFT Re-scoring (Merge + vLLM)

Re-score all factors with the SFT (LoRA fine-tuned) model using merged weights served via vLLM.

**Two-phase approach for maximum throughput:**
1. **Phase A (one-time):** Merge LoRA adapter into base model → save merged fp16 weights
2. **Phase B:** Serve merged model via vLLM → score all factors concurrently

| | |
|---|---|
| **Base model** | `deepseek-ai/DeepSeek-R1-Distill-Qwen-14B` |
| **Adapter** | `models/sentiment_lora_opus/` |
| **Merged model** | `models/sentiment_merged_opus/` |
| **Input** | `../task_1/output/factors_scored/{TICKER}/*_factors.json` |
| **Output** | `output/factors_scored_sft/{TICKER}/*_factors.json` |

In [ ]:
import json
import asyncio
import logging
import re
import sys
import time
from collections import Counter
from pathlib import Path

import nest_asyncio
from openai import AsyncOpenAI
from tqdm.notebook import tqdm

# Allow nested event loops in Jupyter
nest_asyncio.apply()

# ── Paths ────────────────────────────────────────────────────────────────
_cwd = Path.cwd()
TASK1_DIR = (_cwd.parent / "task_1").resolve() if (_cwd.parent / "task_1").exists() else (_cwd / ".." / "task_1").resolve()

FACTORS_SCORED_DIR = TASK1_DIR / "output" / "factors_scored"
TICKER_MAPPING_PATH = TASK1_DIR / "ticker_mapping.json"

SFT_SCORED_DIR = _cwd / "output" / "factors_scored_sft"
LORA_DIR = _cwd / "models" / "sentiment_lora_opus"
MERGED_DIR = _cwd / "models" / "sentiment_merged_opus"

SFT_SCORED_DIR.mkdir(parents=True, exist_ok=True)

# ── Constants ────────────────────────────────────────────────────────────
MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"
SFT_MODEL_TAG = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B+sft_lora"
LABEL_ORDER = ["very_negative", "negative", "neutral", "positive", "very_positive"]
VALID_LABELS = set(LABEL_ORDER)

# ── vLLM server config ──────────────────────────────────────────────────
VLLM_BASE_URL = "http://127.0.0.1:8000/v1"
VLLM_API_KEY = "local"
VLLM_MODEL_NAME = "sentiment_merged_opus"  # adjust to match what vLLM reports at startup
MAX_CONCURRENT = 64  # concurrent in-flight requests to vLLM

# ── Ticker -> sub-sector mapping (needed to reproduce training prompt) ───
with open(TICKER_MAPPING_PATH) as f:
    _ticker_mapping = json.load(f)
TICKER_SUBSECTOR = {}
for subsector, tickers in _ticker_mapping.items():
    for t in tickers:
        TICKER_SUBSECTOR[t] = subsector

# ── Logging ──────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger("task_2_5")

# ── Verify paths ─────────────────────────────────────────────────────────
assert FACTORS_SCORED_DIR.exists(), f"Input directory not found: {FACTORS_SCORED_DIR}"
assert LORA_DIR.exists(), f"LoRA adapter not found: {LORA_DIR}"
print(f"Input factors:  {FACTORS_SCORED_DIR}")
print(f"Output SFT:     {SFT_SCORED_DIR}")
print(f"LoRA adapter:   {LORA_DIR}")
print(f"Merged model:   {MERGED_DIR}")
print(f"Tickers:        {len(TICKER_SUBSECTOR)}")

## Phase A: Merge LoRA into Base Model (One-Time)

Load the base model in **fp16** (NOT 4-bit), apply the LoRA adapter, merge weights permanently, and save.

**Important:** Do NOT merge into a 4-bit model — it corrupts the weights. We load in fp16, merge, then save.
The merged fp16 model is ~28GB and fits on A100 (40GB) or A6000 (48GB).

**Skip this cell if `models/sentiment_merged/` already exists.**

In [ ]:
if MERGED_DIR.exists() and (MERGED_DIR / "config.json").exists():
    print(f"Merged model already exists at {MERGED_DIR}, skipping merge.")
else:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel

    print("Loading base model in fp16 (this requires ~28GB RAM)...")
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )

    print("Loading LoRA adapter...")
    base_model = PeftModel.from_pretrained(base_model, str(LORA_DIR))

    print("Merging LoRA weights into base model...")
    merged_model = base_model.merge_and_unload()

    print(f"Saving merged model to {MERGED_DIR}...")
    MERGED_DIR.mkdir(parents=True, exist_ok=True)
    merged_model.save_pretrained(str(MERGED_DIR))

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer.save_pretrained(str(MERGED_DIR))

    # Free GPU memory
    del merged_model, base_model
    torch.cuda.empty_cache()

    print("Merge complete. Merged model saved.")
    print(f"Files: {list(MERGED_DIR.iterdir())}")

## Phase B: Score Factors via vLLM

**Before running the cells below, start vLLM in a separate terminal:**

```bash
vllm serve models/sentiment_merged_opus/ \
  --dtype auto \
  --gpu-memory-utilization 0.90 \
  --max-model-len 2048 \
  --port 8000
```

Then run the cells below to score all factors concurrently.

In [ ]:
# ── System prompt (matches SFT training format exactly) ──────────────────
SFT_SYSTEM_PROMPT = (
    "You are a financial analyst specializing in SEC filing analysis. "
    "Classify the sentiment of the given factor and respond with JSON only."
)


def format_evidence_text(evidence_list: list[dict]) -> str:
    """Flatten evidence array into a single text string.

    Reproduces the same format used when building the SFT training dataset
    (task_2_1_annotation_dataset.ipynb, Cell 5).
    """
    parts = []
    for ev in evidence_list:
        text = ev.get("text", "").strip()
        if text:
            section = ev.get("section", "")
            subsection = ev.get("subsection", "")
            source = f" [{section}/{subsection}]" if section else ""
            parts.append(f"{text}{source}")
    return " | ".join(parts) if parts else ""


def build_user_prompt(
    factor: dict,
    ticker: str,
    form: str,
    sub_sector: str,
) -> str:
    """Build the user prompt for a single factor.

    Matches the format from task_2_1_annotation_dataset.ipynb (format_user_prompt).
    """
    evidence = format_evidence_text(factor.get("evidence", []))
    # Truncate evidence if too long (same 1500 char limit as training)
    if len(evidence) > 1500:
        evidence = evidence[:1500] + "..."

    return (
        f"Given the following factor extracted from a {form} filing "
        f"for {ticker} ({sub_sector} sub-sector), classify the sentiment.\n\n"
        f"Category: {factor['category']}\n"
        f"Factor: {factor['key']}\n"
        f"Summary: {factor['summary']}\n"
        f"Evidence: {evidence}\n\n"
        f"Classify the sentiment as one of: very_negative, negative, neutral, positive, very_positive.\n"
        f"Provide a brief rationale and confidence score (0.0-1.0).\n"
        f"Respond with JSON only."
    )


def parse_json_response(text: str) -> dict | None:
    """Parse JSON from model output, handling think blocks and markdown fences."""
    # Strip <think>...</think> block if present
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

    # Strip markdown code fences
    text = re.sub(r"^```(?:json)?\s*\n?", "", text, flags=re.MULTILINE)
    text = re.sub(r"\n?```\s*$", "", text, flags=re.MULTILINE)
    text = text.strip()

    # Try direct parse
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Fallback: find first { to last }
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        try:
            return json.loads(text[start : end + 1])
        except json.JSONDecodeError:
            pass

    return None


print("Prompt functions defined:")
print("  - format_evidence_text(evidence_list)")
print("  - build_user_prompt(factor, ticker, form, sub_sector)")
print("  - parse_json_response(text)")

### Discover Factor Files & Resume Filter

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# TICKER FILTER — Set your assigned tickers here to split work across team
# ══════════════════════════════════════════════════════════════════════════
# Leave as None to process ALL tickers.
MY_TICKERS = None  # e.g. {"AAL", "ADT", "ALK", ...}

all_factor_files = sorted(FACTORS_SCORED_DIR.rglob("*_factors.json"))
if MY_TICKERS is not None:
    all_factor_files = [f for f in all_factor_files if f.parent.name in MY_TICKERS]
    print(f"TICKER FILTER ACTIVE: {len(MY_TICKERS)} tickers")
print(f"Total factor files: {len(all_factor_files)}")

# Resume-safe filter
pending_files = []
already_done = 0
for ff in all_factor_files:
    out_file = SFT_SCORED_DIR / ff.parent.name / ff.name
    if out_file.exists():
        already_done += 1
    else:
        pending_files.append(ff)
print(f"Already done: {already_done}")
print(f"Pending:      {len(pending_files)}")

### Build All Prompts

Pre-build all (factor, prompt) pairs for pending files upfront so we can fire them all at vLLM concurrently.

In [ ]:
# Build a flat list of all (factor_file, factor_index, messages) to score
all_tasks = []

for ff in tqdm(pending_files, desc="Building prompts"):
    with open(ff) as f:
        data = json.load(f)
    ticker = data.get("ticker", ff.parent.name)
    form = data.get("form", "")
    sub_sector = TICKER_SUBSECTOR.get(ticker, "unknown")

    for idx, factor in enumerate(data.get("factors", [])):
        user_prompt = build_user_prompt(factor, ticker, form, sub_sector)
        messages = [
            {"role": "system", "content": SFT_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ]
        all_tasks.append({
            "file": ff,
            "factor_idx": idx,
            "factor_key": factor.get("key", ""),
            "messages": messages,
        })

print(f"Total scoring tasks: {len(all_tasks):,}")

### Async Scoring Loop

Fire all prompts concurrently at vLLM with a semaphore to cap in-flight requests at `MAX_CONCURRENT`.

In [ ]:
client = AsyncOpenAI(base_url=VLLM_BASE_URL, api_key=VLLM_API_KEY)
semaphore = asyncio.Semaphore(MAX_CONCURRENT)

results = {}  # file_path_str -> {factor_idx -> sentiment_dict}

async def score_one(task: dict) -> None:
    """Score a single factor via vLLM API."""
    async with semaphore:
        try:
            response = await client.chat.completions.create(
                model=VLLM_MODEL_NAME,
                messages=task["messages"],
                temperature=0.1,
                max_tokens=256,
            )
            text = response.choices[0].message.content or ""
            parsed = parse_json_response(text)

            if parsed is not None:
                label = (parsed.get("label") or parsed.get("sentiment", "")).strip().lower()
                if label in VALID_LABELS:
                    fpath = str(task["file"])
                    if fpath not in results:
                        results[fpath] = {}
                    results[fpath][task["factor_idx"]] = {
                        "label": label,
                        "rationale": parsed.get("rationale", ""),
                        "confidence": min(1.0, max(0.0, float(parsed.get("confidence", 0.5)))),
                    }
        except Exception as e:
            log.warning(f"  Failed: {task['factor_key']} — {e}")

async def run_all():
    tasks_coros = [score_one(t) for t in all_tasks]
    # Use tqdm for progress tracking
    for coro in tqdm(asyncio.as_completed(tasks_coros), total=len(tasks_coros), desc="Scoring via vLLM"):
        await coro

t0 = time.time()
await run_all()
elapsed = time.time() - t0

scored_count = sum(len(v) for v in results.values())
print(f"\nScored {scored_count:,} factors in {elapsed:.1f}s ({scored_count/max(1,elapsed):.1f} factors/sec)")

### Save Results

Write scored factors back to JSON files, preserving the original schema with updated sentiment and model tag.

In [ ]:
saved_filings = 0
total_scored = 0
total_failed = 0

for ff in tqdm(pending_files, desc="Saving results"):
    with open(ff) as f:
        data = json.load(f)

    fpath = str(ff)
    file_results = results.get(fpath, {})

    for idx, factor in enumerate(data.get("factors", [])):
        sentiment = file_results.get(idx)
        if sentiment is not None:
            factor["sentiment"] = sentiment
            total_scored += 1
        else:
            factor["sentiment"] = None
            total_failed += 1

    data["model"] = SFT_MODEL_TAG

    out_dir = SFT_SCORED_DIR / ff.parent.name
    out_dir.mkdir(parents=True, exist_ok=True)
    out_file = out_dir / ff.name
    with open(out_file, "w") as f:
        json.dump(data, f, indent=2)
    saved_filings += 1

print(f"\nSaved {saved_filings} filings")
print(f"Factors scored: {total_scored:,}")
print(f"Factors failed: {total_failed:,}")
print(f"Success rate:   {total_scored / max(1, total_scored + total_failed) * 100:.1f}%")

## Step 6: Validation — Label Distribution Comparison

Compare the label distributions between the base model (task_1 scored) and the SFT model
to verify the re-scoring produced reasonable results.

In [ ]:
def collect_labels(base_dir: Path) -> Counter:
    """Collect sentiment label counts from all factor files in a directory."""
    counts = Counter()
    null_count = 0
    for fpath in sorted(base_dir.rglob("*_factors.json")):
        with open(fpath) as f:
            data = json.load(f)
        for fac in data.get("factors", []):
            sentiment = fac.get("sentiment")
            if isinstance(sentiment, dict) and sentiment.get("label"):
                counts[sentiment["label"]] += 1
            else:
                null_count += 1
    return counts, null_count


# ── Base model labels ────────────────────────────────────────────────────
print("Collecting base model labels...")
base_counts, base_nulls = collect_labels(FACTORS_SCORED_DIR)

# ── SFT model labels ────────────────────────────────────────────────────
print("Collecting SFT model labels...")
sft_counts, sft_nulls = collect_labels(SFT_SCORED_DIR)

# ── Print comparison ─────────────────────────────────────────────────────
base_total = sum(base_counts.values())
sft_total = sum(sft_counts.values())

print(f"\n{'='*60}")
print(f"LABEL DISTRIBUTION COMPARISON")
print(f"{'='*60}")
print(f"{'Label':<16s} | {'Base':>8s} {'%':>6s} | {'SFT':>8s} {'%':>6s} | {'Delta':>6s}")
print(f"{'-'*16}-+-{'-'*15}-+-{'-'*15}-+-{'-'*6}")

for label in LABEL_ORDER:
    bc = base_counts.get(label, 0)
    sc = sft_counts.get(label, 0)
    bp = bc / max(1, base_total) * 100
    sp = sc / max(1, sft_total) * 100
    delta = sp - bp
    print(f"{label:<16s} | {bc:>8,d} {bp:>5.1f}% | {sc:>8,d} {sp:>5.1f}% | {delta:>+5.1f}%")

print(f"{'-'*16}-+-{'-'*15}-+-{'-'*15}-+-{'-'*6}")
print(f"{'TOTAL':<16s} | {base_total:>8,d}       | {sft_total:>8,d}       |")
print(f"{'NULL/failed':<16s} | {base_nulls:>8,d}       | {sft_nulls:>8,d}       |")

# ── Quick sanity check on a few samples ──────────────────────────────────
print(f"\n{'='*60}")
print("SAMPLE COMPARISON (first 5 factors from first SFT file)")
print(f"{'='*60}")

sft_files = sorted(SFT_SCORED_DIR.rglob("*_factors.json"))
if sft_files:
    sample_sft = sft_files[0]
    # Find the corresponding base file
    ticker = sample_sft.parent.name
    sample_base = FACTORS_SCORED_DIR / ticker / sample_sft.name

    with open(sample_sft) as f:
        sft_data = json.load(f)
    with open(sample_base) as f:
        base_data = json.load(f)

    print(f"File: {sample_sft.name}")
    print(f"Ticker: {sft_data['ticker']}, Form: {sft_data['form']}, Date: {sft_data['filing_date']}")
    print()

    for i, (bf, sf) in enumerate(zip(base_data["factors"][:5], sft_data["factors"][:5])):
        base_label = bf.get("sentiment", {}).get("label", "N/A") if isinstance(bf.get("sentiment"), dict) else "N/A"
        sft_label = sf.get("sentiment", {}).get("label", "N/A") if isinstance(sf.get("sentiment"), dict) else "N/A"
        match = "==" if base_label == sft_label else "!="
        print(f"  [{i}] {bf['key']:<30s}  base={base_label:<14s} sft={sft_label:<14s} {match}")
else:
    print("No SFT output files found yet.")